In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import google.protobuf

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("Protobuf:", google.protobuf.__version__)

print("tfds.load exists:", hasattr(tfds, "load"))

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE = 32

dataset = tfds.load(
    "celeb_a",
    split="train",
    shuffle_files=True
)

print(dataset)

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE = 32
TRAIN_SAMPLES = 20000

def preprocess(example):
    image = example["image"]

    # Resize image
    image = tf.image.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    # Normalize pixel values to [0, 1]
    image = tf.cast(image, tf.float32) / 255.0

    return image


train_dataset = (
    dataset
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(5000)
    .take(TRAIN_SAMPLES)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(train_dataset)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [ ]:
images = next(iter(train_dataset))

plt.figure(figsize=(10, 10))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(images[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
INPUT_DIM = (64, 64, 3)
LATENT_DIM = 200

In [ ]:
class Sampling(tf.keras.layers.Layer):
    """Uses z_mean and z_log_var to sample the latent vector z."""

    def call(self, inputs):
        z_mean, z_log_var = inputs

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        epsilon = tf.random.normal(shape=(batch, dim))

        z = z_mean + tf.exp(0.5 * z_log_var) * epsilon

        return z

In [ ]:
def build_encoder(input_dim, latent_dim):

    # Input layer
    encoder_input = layers.Input(
        shape=input_dim,
        name="encoder_input"
    )

    x = encoder_input

    # Convolution Layer 1
    x = layers.Conv2D(
        32,
        kernel_size=3,
        strides=2,
        padding="same",
        name="encoder_conv_1"
    )(x)

    x = layers.LeakyReLU()(x)


    # Convolution Layer 2
    x = layers.Conv2D(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="encoder_conv_2"
    )(x)

    x = layers.LeakyReLU()(x)


    # Convolution Layer 3
    x = layers.Conv2D(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="encoder_conv_3"
    )(x)

    x = layers.LeakyReLU()(x)


    # Convolution Layer 4
    x = layers.Conv2D(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="encoder_conv_4"
    )(x)

    x = layers.LeakyReLU()(x)


    # Store shape for decoder
    shape_before_flattening = tuple(x.shape[1:])


    # Flatten
    x = layers.Flatten(
        name="encoder_flatten"
    )(x)


    # Mean of latent distribution
    z_mean = layers.Dense(
        latent_dim,
        name="z_mean"
    )(x)


    # Log variance of latent distribution
    z_log_var = layers.Dense(
        latent_dim,
        name="z_log_var"
    )(x)


    # Sample latent vector
    z = Sampling()(
        [z_mean, z_log_var]
    )


    # Create encoder model
    encoder = keras.Model(
        encoder_input,
        [z_mean, z_log_var, z],
        name="encoder"
    )

    return encoder, shape_before_flattening

In [ ]:
encoder, shape_before_flattening = build_encoder(
    INPUT_DIM,
    LATENT_DIM
)

encoder.summary()

print("\nShape before flattening:", shape_before_flattening)

In [ ]:
def build_decoder(latent_dim, shape_before_flattening):

    # Input latent vector
    decoder_input = layers.Input(
        shape=(latent_dim,),
        name="decoder_input"
    )

    # Calculate number of units
    # Convert to Python int for Keras compatibility
    units = int(np.prod(shape_before_flattening))

    print("Shape before flattening:", shape_before_flattening)
    print("Units:", units)
    print("Type of units:", type(units))

    # Expand latent vector
    x = layers.Dense(
        units,
        name="decoder_dense"
    )(decoder_input)

    # Reshape
    x = layers.Reshape(
        shape_before_flattening,
        name="decoder_reshape"
    )(x)

    # 4x4 -> 8x8
    x = layers.Conv2DTranspose(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="decoder_conv_t_1"
    )(x)

    x = layers.LeakyReLU()(x)

    # 8x8 -> 16x16
    x = layers.Conv2DTranspose(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="decoder_conv_t_2"
    )(x)

    x = layers.LeakyReLU()(x)

    # 16x16 -> 32x32
    x = layers.Conv2DTranspose(
        64,
        kernel_size=3,
        strides=2,
        padding="same",
        name="decoder_conv_t_3"
    )(x)

    x = layers.LeakyReLU()(x)

    # 32x32 -> 64x64
    x = layers.Conv2DTranspose(
        32,
        kernel_size=3,
        strides=2,
        padding="same",
        name="decoder_conv_t_4"
    )(x)

    x = layers.LeakyReLU()(x)

    # Final output
    decoder_output = layers.Conv2D(
        3,
        kernel_size=3,
        padding="same",
        activation="sigmoid",
        name="decoder_output"
    )(x)

    # Create decoder
    decoder = keras.Model(
        decoder_input,
        decoder_output,
        name="decoder"
    )

    return decoder

In [ ]:
decoder = build_decoder(
    LATENT_DIM,
    shape_before_flattening
)

decoder.summary()

In [ ]:
class VAE(keras.Model):

    def __init__(self, encoder, decoder, beta=0.001, **kwargs):
        super().__init__(**kwargs)

        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

        # Metrics
        self.total_loss_tracker = keras.metrics.Mean(
            name="total_loss"
        )

        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss"
        )

        self.kl_loss_tracker = keras.metrics.Mean(
            name="kl_loss"
        )


    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker
        ]


    def train_step(self, data):

        with tf.GradientTape() as tape:

            # -----------------------------
            # Encoder
            # -----------------------------
            z_mean, z_log_var, z = self.encoder(
                data,
                training=True
            )

            # -----------------------------
            # Decoder
            # -----------------------------
            reconstruction = self.decoder(
                z,
                training=True
            )

            # -----------------------------
            # Reconstruction Loss
            # Mean Squared Error
            # -----------------------------
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.square(data - reconstruction),
                    axis=(1, 2, 3)
                )
            )

            # -----------------------------
            # KL Divergence Loss
            # -----------------------------
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1
                    + z_log_var
                    - tf.square(z_mean)
                    - tf.exp(z_log_var),
                    axis=1
                )
            )

            # -----------------------------
            # Total Loss
            # -----------------------------
            total_loss = (
                reconstruction_loss
                + self.beta * kl_loss
            )

        # Calculate gradients
        grads = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        # Update weights
        self.optimizer.apply_gradients(
            zip(grads, self.trainable_weights)
        )

        # Update metrics
        self.total_loss_tracker.update_state(
            total_loss
        )

        self.reconstruction_loss_tracker.update_state(
            reconstruction_loss
        )

        self.kl_loss_tracker.update_state(
            kl_loss
        )

        return {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result()
        }

In [ ]:
LEARNING_RATE = 0.0005

vae = VAE(
    encoder=encoder,
    decoder=decoder,
    beta=0.001
)

vae.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    )
)

print("VAE model created successfully!")

In [ ]:
EPOCHS = 20

history = vae.fit(
    train_dataset,
    epochs=EPOCHS
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history.history["total_loss"],
    marker="o",
    label="Total Loss"
)

plt.plot(
    history.history["reconstruction_loss"],
    marker="o",
    label="Reconstruction Loss"
)

plt.plot(
    history.history["kl_loss"],
    marker="o",
    label="KL Divergence Loss"
)

plt.title("VAE Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
def plot_reconstructions(vae, dataset, n_to_show=8):

    # Get one batch of images
    example_images = next(iter(dataset))

    # Encode images
    z_mean, z_log_var, z = vae.encoder.predict(
        example_images,
        verbose=0
    )

    # Decode latent vectors
    reconstructed_images = vae.decoder.predict(
        z,
        verbose=0
    )

    plt.figure(figsize=(16, 5))

    for i in range(n_to_show):

        # Original image
        plt.subplot(2, n_to_show, i + 1)

        plt.imshow(example_images[i])
        plt.title("Original")
        plt.axis("off")


        # Reconstructed image
        plt.subplot(2, n_to_show, i + n_to_show + 1)

        plt.imshow(reconstructed_images[i])
        plt.title("Reconstructed")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_reconstructions(
    vae,
    train_dataset,
    n_to_show=8
)

In [ ]:
def generate_faces(vae, n_faces=20):

    # Generate random latent vectors
    z_sample = np.random.normal(
        loc=0,
        scale=1,
        size=(n_faces, LATENT_DIM)
    ).astype("float32")

    # Generate faces
    generated_images = vae.decoder.predict(
        z_sample,
        verbose=0
    )

    # Display faces
    plt.figure(figsize=(15, 8))

    for i in range(n_faces):

        plt.subplot(4, 5, i + 1)

        plt.imshow(generated_images[i])
        plt.axis("off")

    plt.suptitle(
        "Generated Synthetic Faces",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

In [ ]:
generate_faces(
    vae,
    n_faces=20
)

In [ ]:
EPOCHS = 30

history_2 = vae.fit(
    train_dataset,
    epochs=EPOCHS
)

In [ ]:
generate_faces(
    vae,
    n_faces=20
)

In [ ]:
plot_reconstructions(
    vae,
    train_dataset,
    n_to_show=8
)

In [ ]:
def plot_latent_distributions(
    vae,
    dataset,
    n_dims=20
):

    # Get images
    example_images = next(iter(dataset))

    # Encode images
    z_mean, z_log_var, z = vae.encoder.predict(
        example_images,
        verbose=0
    )

    plt.figure(figsize=(15, 12))

    for i in range(n_dims):

        plt.subplot(4, 5, i + 1)

        plt.hist(
            z_mean[:, i],
            bins=20,
            alpha=0.6,
            label="z_mean"
        )

        plt.hist(
            z[:, i],
            bins=20,
            alpha=0.6,
            label="z_sample"
        )

        plt.title(f"Latent Dimension {i + 1}")
        plt.legend(fontsize=7)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_latent_distributions(
    vae,
    train_dataset,
    n_dims=20
)

The Variational Autoencoder (VAE) was successfully implemented and trained on the CelebA face dataset. The encoder learned a 200-dimensional probabilistic latent representation using mean and log-variance vectors, while the decoder reconstructed images from sampled latent vectors. The model successfully generated new synthetic face-like images and reconstructed input images. Although the generated faces were somewhat blurry, facial structures became more recognizable with additional training, demonstrating that the VAE learned meaningful patterns from the dataset.